In [12]:
import json

#need to come back
with open("20_000_qwen3_MedQA_rephrase.jsonl", 'r') as infile, open("synth_medqa-20_000.json", 'w') as outfile:
    data = json.load(infile)

    for key, value in data.items():
        if value.get("status") == "success" and "medQA_rephrase" in value:
            rephrase_data = value["medQA_rephrase"]
            label_map = {"a": 0, "b": 1, "c": 2, "d": 3, "e": 4}
            correct = rephrase_data.get("cop", "").lower()
            
            if correct in label_map:
                correct = label_map[correct]
                # build options list
                options = [
                    rephrase_data.get("opa", "").lstrip("a) "),
                    rephrase_data.get("opb", "").lstrip("b) "),
                    rephrase_data.get("opc", "").lstrip("c) "),
                    rephrase_data.get("opd", "").lstrip("d) "),
                    rephrase_data.get("ope", "").lstrip("e) "),
                ]

                if correct == 4:
                    # drop the first option (a), keep b–e → now 4 options
                    options = options[1:]  
                    label = 3  # because "e" becomes last in the truncated list
                else:
                    # keep first 4 options only
                    options = options[:4]
                    # label = {"a": 0, "b": 1, "c": 2, "d": 3}.get(correct, -1)

                output_record = {
                    "id": f"dev-{value['id']:05d}",
                    "sent1": rephrase_data.get("question", ""),
                    "sent2": "",
                    "ending0": options[0],
                    "ending1": options[1],
                    "ending2": options[2],
                    "ending3": options[3],
                    "label": label
                }

                outfile.write(json.dumps(output_record) + "\n")


In [11]:
value['medQA_rephrase']

{'question': 'A new pegylated drug is being developed with therapeutic pathways linked to hepatitis and neutropenia. Based on the pharmacokinetic advantages and clinical applications of existing pegylated therapies, which of the following therapeutic areas is this drug most likely targeting?',
 'opa': 'Chronic viral infections',
 'opb': 'Autoimmune disorders',
 'opc': 'Diabetes mellitus',
 'opd': 'Oncology support',
 'ope': 'Cardiovascular disease',
 'reasoning': 'a) Chronic viral infections is correct because pegylation enhances drug stability and prolongs antiviral efficacy, as seen in peginterferon alfa-2a for hepatitis C. This aligns with the question’s reference to hepatitis. b) Autoimmune disorders is incorrect because pegylated drugs for autoimmune conditions (e.g., adalimumab) target cytokines, not viral replication or neutrophil recovery. c) Diabetes mellitus is incorrect because insulin analogs, while modified for prolonged action, are not pegylated. d) Oncology support is co

In [5]:
data['2']

{'id': 2,
 'qa': "In the context of cellular protein regulation, how might the removal of Orc1 from chromatin influence protein degradation pathways, given the involvement of specific 26S proteasome subunits? The removal of Orc1 from chromatin may facilitate the proteasomal degradation of DVL, a process mediated by the 26S proteasome's non-ATPase 6 and SEM1 subunits. This pathway suggests Orc1's removal could trigger a cascade affecting protein stability, potentially impacting cell cycle progression or Wnt/β-catenin signaling, where DVL plays a crucial role. This proteasome-mediated degradation highlights the intricate balance between protein synthesis and degradation in maintaining cellular homeostasis.",
 'medQA_similar': ["A young immigrant girl presents with low-grade fever, sore throat, painful swallowing, and difficulty in breathing. Her voice is unusually nasal and her swollen neck gives the impression of “bull's neck”. On examination, a large gray membrane is noticed on the oro

In [3]:
# import json
# import random

# # Set a fixed seed for reproducibility
# random.seed(42)

# # Load datasets
# with open('synth_pubmedqa.jsonl', 'r') as f:
#     synth_data = [json.loads(line) for line in f]

# with open('/home/id.aau.dk/kr75cs/kgqa/LinkBERT/data/seqcls/pubmedqa_hf/train.json', 'r') as f:
#     gold_data = [json.loads(line) for line in f]

# # Optional: oversample gold data
# oversampled_gold = gold_data * 10

# # Combine datasets
# combined_data = synth_data + oversampled_gold

# # Shuffle combined data
# random.shuffle(combined_data)

# # Save to a new JSONL file
# with open('combined_pubmedqa.jsonl', 'w') as f:
#     for item in combined_data:
#         f.write(json.dumps(item) + '\n')

# print(f"Combined dataset saved. Total examples: {len(combined_data)}")


Combined dataset saved. Total examples: 24387


In [1]:
# import json
# import random

# # Set a fixed seed for reproducibility
# random.seed(42)

# # Load datasets
# with open('synth_pubmedqa.jsonl', 'r') as f:
#     synth_data = [json.loads(line) for line in f]

# with open('/home/id.aau.dk/kr75cs/kgqa/LinkBERT/data/seqcls/pubmedqa_hf/train.json', 'r') as f:
#     gold_data = [json.loads(line) for line in f]

# # Optional: oversample gold data
# oversampled_gold = gold_data * 10

# # Combine datasets
# combined_data = synth_data[:5000] + oversampled_gold

# # Shuffle combined data
# random.shuffle(combined_data)

# # Save to a new JSONL file
# with open('combined_pubmedqa_5000.jsonl', 'w') as f:
#     for item in combined_data:
#         f.write(json.dumps(item) + '\n')

# print(f"Combined dataset saved. Total examples: {len(combined_data)}")


Combined dataset saved. Total examples: 9500


In [1]:
import json
import random

# Set a fixed seed for reproducibility
random.seed(42)

# Paths
synth_path = "/home/id.aau.dk/kr75cs/kgqa/LinkBERT/data/mc/medqa_usmle_hf/synth_medqa-20_000.json"
gold_path = "/home/id.aau.dk/kr75cs/kgqa/LinkBERT/data/mc/medqa_usmle_hf/train.json"

# Load datasets
with open(gold_path, 'r') as f:
    gold_data = [json.loads(line) for line in f]

with open(synth_path, 'r') as f:
    synth_data = [json.loads(line) for line in f]

# Write synthetic subsets
for size in [100, 500, 1_000, 5_000, 10_000]:
    out_path = f"/home/id.aau.dk/kr75cs/kgqa/LinkBERT/data/mc/medqa_usmle_hf/synth_{size}.json"
    with open(out_path, 'w') as f:
        for item in synth_data[:size]:
            f.write(json.dumps(item) + "\n")

# Write gold subsets
for size in [100, 500, 2_500, 1_000, 5_000, 7_500]:
    out_path = f"/home/id.aau.dk/kr75cs/kgqa/LinkBERT/data/mc/medqa_usmle_hf/train_{size}.json"
    with open(out_path, 'w') as f:
        for item in gold_data[:size]:
            f.write(json.dumps(item) + "\n")


In [1]:
import json
import random

# Set a fixed seed for reproducibility
random.seed(42)

# Load datasets
with open('synth_pubmedqa.jsonl', 'r') as f:
    synth_data = [json.loads(line) for line in f]
    
with open('/home/id.aau.dk/kr75cs/kgqa/LinkBERT/data/seqcls/pubmedqa_hf/eval_synth_1000.json', 'w') as f:
    for item in synth_data[-1_000:]:
        f.write(json.dumps(item) + '\n')